[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UoA-eResearch/embedding-llms-qualitative-data-workshop/blob/main/notebooks/05-visual-extraction.ipynb)

# Notebook 05 — Visual Feature Extraction

**Duration:** ~25 minutes

In notebooks 03 and 04 you extracted information and generated themes from legislative text. In this notebook you apply the **exact same pipeline** to images.

The dataset is a curated selection of historical protest and petition images from **Archives New Zealand**, accessed via Wikimedia Commons. All images are in the public domain under New Zealand's NZGOAL framework.

You will:
1. Send an image to a vision-capable LLM and extract structured features as JSON
2. Compare the image keywords against the parliamentary committee keyword sets from notebook 04 using Jaccard similarity

The point: the data type changed. The methodology did not.

## Setup

Run this cell first. Paste your Groq API key between the quotes.

Note: this notebook uses a **vision model** — a model that can read images as well as text.

In [ ]:
# ============================================================
# SETUP CELL — Run this once at the start of every notebook
# ============================================================

!pip install groq requests lxml Pillow
import os, json, base64, requests, io
from groq import Groq
from lxml import etree
from PIL import Image
from IPython.display import Image as IPImage, display

os.environ["GROQ_API_KEY"] = "paste_your_key_here"   # <-- paste your key here
client = Groq(api_key=os.environ["GROQ_API_KEY"])
TEXT_MODEL = "llama-3.3-70b-versatile"
VISION_MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"

print("Setup complete.")

## Load an image from Archives New Zealand

We fetch images from the **Wikimedia Commons API** — a public repository of freely licensed media. Archives New Zealand has contributed thousands of historical images to the collection.

The curated corpus for this workshop contains three images:

| Image | Year | Subject |
|-------|------|---------|
| Prosecution of Strikers: The 1913 Black List | 1913 | Labour disputes, blacklisting of workers |
| Petition of Unemployment | 1870 | Petition to government about unemployment |
| Petition against Steam Trams | 1879 | Community petition opposing steam tram routes |

Run the cells below. They fetch an image, resize it for the API, and display it in your notebook.

In [ ]:
# Choose an image from the curated corpus.
# You can change this to any of the three options below.
#
# Options:
#   "File:Prosecution_of_Strikers-_The_1913_Black_List_(29788240653).jpg"
#   "File:1870_Petition_of_Unemployment_(9926603304).jpg"
#   "File:1879_Petition_against_Steam_Trams_(29078906095).jpg"

image_title = "File:Prosecution_of_Strikers-_The_1913_Black_List_(29788240653).jpg"

# Fetch the image URL from the Wikimedia Commons API
api_url = "https://commons.wikimedia.org/w/api.php"
params = {
    "action": "query",
    "titles": image_title,
    "prop": "imageinfo",
    "iiprop": "url|size|extmetadata",
    "format": "json"
}
headers = {
    "User-Agent": "UoA-eResearch-LLM-Workshop/1.0 (researchdata@auckland.ac.nz)"
}

api_response = requests.get(api_url, params=params, headers=headers)
pages = api_response.json()["query"]["pages"]
page = list(pages.values())[0]
image_info = page["imageinfo"][0]
image_url = image_info["url"]

# Extract the archival description (we will use this later for validation)
metadata = image_info.get("extmetadata", {})
archival_description = metadata.get("ImageDescription", {}).get("value", "No description available.")

print(f"Image: {image_title.split(':')[1]}")
print(f"Source URL: {image_url[:80]}...")
print(f"Original size: {image_info['width']}x{image_info['height']}")

In [ ]:
# Download the image, resize to width=400 (to stay within API limits), and display it
img_response = requests.get(image_url, headers=headers)
img = Image.open(io.BytesIO(img_response.content))

# Resize
new_width = 400
ratio = new_width / img.width
new_height = int(img.height * ratio)
img_resized = img.resize((new_width, new_height))

# Convert to base64 for the API call
buffer = io.BytesIO()
img_resized.save(buffer, format="JPEG")
img_base64 = base64.b64encode(buffer.getvalue()).decode("utf-8")

# Display the image in the notebook
display(IPImage(data=buffer.getvalue(), format="jpeg"))
print(f"\nResized to {new_width}x{new_height} for API.")

You should see a historical document displayed above. Take a moment to look at it before we ask the model to analyse it.

## Where this fits in the research workflow

> **Research context:** The published study on NZ legislative networks (Ardekani et al., 2026) worked entirely with text. But the extraction and validation logic you have been practising works on any unstructured source. Images are just another kind of unstructured data. The pipeline is identical: extract structured features, generate keywords, validate against an external reference.

In this notebook you are doing:
- **Extraction** (Step 2) — pulling structured JSON from an image instead of from XML
- **Interpretation** (Step 6) — validating image keywords against committee domains using Jaccard similarity

## Exercise a — Structured JSON extraction from an image

In notebook 03 you asked the LLM to extract Act references from text as JSON. Now you do the same thing with an image — ask the model to describe what it sees and return the results in a structured format.

The prompt asks the model to fill in these fields:
- **scene_description** — what is happening in the image
- **political_claim** — what political argument or demand is being made
- **visible_text** — any text that appears in the image
- **symbols** — visual symbols, logos, flags, or iconography
- **keywords** — 10 to 15 keywords describing the image's content (this feeds into exercise b)
- **confidence** — how confident the model is in its analysis (high / medium / low)

Run the cell below.

In [ ]:
vision_response = client.chat.completions.create(
    model=VISION_MODEL,
    temperature=0,
    messages=[
        {
            "role": "system",
            "content": "You are a research assistant analysing historical archival images from New Zealand."
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": """Analyse this historical image from Archives New Zealand.

Extract the following features and return them as a JSON object:
{
  "scene_description": "What is happening in this image?",
  "political_claim": "What political argument or demand is being made?",
  "visible_text": "Any text visible in the image (or 'none' if not legible)",
  "symbols": "Visual symbols, logos, flags, or iconography present",
  "keywords": ["10 to 15 keywords describing the content"],
  "confidence": "high, medium, or low"
}

Return ONLY the JSON object."""
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{img_base64}"
                    }
                }
            ]
        }
    ]
)

vision_raw = vision_response.choices[0].message.content.strip()

# Parse the JSON
try:
    image_features = json.loads(vision_raw)
except json.JSONDecodeError:
    # Strip markdown code fences if present
    cleaned = vision_raw.strip("`").strip()
    if cleaned.startswith("json"):
        cleaned = cleaned[4:].strip()
    image_features = json.loads(cleaned)

# Pretty-print the results
print("Extracted features:")
print("=" * 60)
for key, value in image_features.items():
    if isinstance(value, list):
        print(f"\n{key}:")
        for item in value:
            print(f"  - {item}")
    else:
        print(f"\n{key}: {value}")

Read through the results. Some questions to consider:

- Does the **scene description** match what you see?
- Did the model correctly read any **visible text** in the image, or did it guess?
- Is the **political claim** reasonable, or did the model project something that is not there?
- How would you rate your own confidence compared to the model's?

This is the same critical eye you applied to text extraction in notebook 03. The data type is different. The questions are the same.

## Exercise b — Cross-modal Jaccard validation

In notebook 04 you compared theme keywords against parliamentary committee keyword sets using Jaccard similarity. Now you do the same thing with the keywords the model extracted from the image.

The question: does the model's keyword description of a historical protest image align with any recognisable policy domain?

In [ ]:
# Jaccard similarity function (same as notebook 04)
def jaccard_similarity(set_a, set_b):
    """Calculate Jaccard similarity between two sets of strings."""
    a = set(w.lower() for w in set_a)
    b = set(w.lower() for w in set_b)
    intersection = a & b
    union = a | b
    if not union:
        return 0.0
    return len(intersection) / len(union)

# Parliamentary select committee keyword sets (same as notebook 04)
COMMITTEE_KEYWORDS = {
    "Justice Committee": [
        "criminal", "justice", "court", "offence", "penalty",
        "prosecution", "enforcement", "police", "imprisonment",
        "tribunal", "legal", "rights", "appeal", "sentence", "conviction"
    ],
    "Health Committee": [
        "health", "medical", "treatment", "patient", "clinical",
        "disease", "mental", "disability", "care", "hospital",
        "practitioner", "pharmaceutical", "wellbeing", "public health",
        "safety"
    ],
    "Environment Committee": [
        "environment", "resource", "land", "water", "conservation",
        "climate", "emissions", "biodiversity", "sustainable",
        "pollution", "ecological", "planning", "consent", "impact",
        "natural"
    ],
    "Finance and Expenditure Committee": [
        "financial", "revenue", "tax", "expenditure", "budget",
        "fiscal", "economic", "payment", "fund", "investment",
        "commercial", "income", "cost", "penalty", "levy"
    ],
    "Governance and Administration Committee": [
        "privacy", "information", "data", "personal", "agency",
        "public", "government", "official", "disclosure", "access",
        "transparency", "accountability", "complaint", "commissioner",
        "record"
    ],
    "Social Services and Community Committee": [
        "social", "community", "welfare", "family", "housing",
        "poverty", "employment", "education", "youth", "elderly",
        "disability", "support", "benefit", "vulnerable", "protection"
    ]
}

print("Jaccard function and committee keywords loaded.")

In [ ]:
# Get the image keywords from the extraction
image_keywords = image_features.get("keywords", [])

print(f"Image keywords ({len(image_keywords)}):")
print(f"  {', '.join(image_keywords)}")
print()

# Compare image keywords against each committee
print("Image keywords — Jaccard similarity with each committee:")
print("=" * 60)
image_scores = {}
for committee, kws in COMMITTEE_KEYWORDS.items():
    score = jaccard_similarity(image_keywords, kws)
    image_scores[committee] = score
    bar = "█" * int(score * 40)
    print(f"  {committee:45s} {score:.3f} {bar}")

best_committee = max(image_scores, key=image_scores.get)
print(f"\nBest match: {best_committee} ({image_scores[best_committee]:.3f})")

### Does the match make sense?

The 1913 Black List image is about the prosecution of striking workers — a labour rights and justice issue. It should align most closely with the **Justice Committee** or the **Social Services and Community Committee**.

If you used a different image, the alignment may differ. The 1870 unemployment petition might lean toward Social Services. The 1879 steam trams petition might lean toward Environment.

Think about:
- Does the model's committee match align with what you see in the image?
- If there is a discrepancy, is it because the model misread the image, or because the committee keywords do not capture the topic well?
- Would a human researcher categorise this image the same way?

### Compare against archival metadata

The Wikimedia Commons API provides the original archival description for each image. This is metadata written by human archivists — an independent reference we can compare against.

This is the same logic as the human spot-check from notebook 03: compare what the model produced against a trusted external source.

In [ ]:
# Show the archival metadata alongside the model's description
print("ARCHIVAL METADATA (written by human archivists):")
print("=" * 60)
print(archival_description)
print()
print("MODEL'S DESCRIPTION:")
print("=" * 60)
print(image_features.get("scene_description", "(not available)"))
print()
print("MODEL'S POLITICAL CLAIM:")
print("=" * 60)
print(image_features.get("political_claim", "(not available)"))
print()
print("Questions to consider:")
print("  1. Does the model's description match the archival record?")
print("  2. Did the model add details not in the metadata (hallucination)?")
print("  3. Did the metadata contain information the model missed?")

> "You used the same extraction and validation pipeline on an image that you used on text. The data type changed. The methodology did not."

## Optional: Try a different image

If you have time, go back to the image selection cell at the top and change `image_title` to one of the other two images. Run all the cells again from that point and see how the results differ.

You can also try these take-home exercises after the workshop:
- Run the extraction with a **positioned persona** (e.g. "You are a labour historian") and compare the Jaccard scores
- Use **adversarial prompting** — challenge the model's scene description
- Try an image from a completely different domain and see which committee it aligns with

## What you accomplished

In this notebook you:

- Fetched a historical image from Archives New Zealand via the Wikimedia Commons API
- Extracted structured features (scene, political claim, text, symbols, keywords) from the image using a vision-capable LLM
- Validated the image keywords against parliamentary committee domains using **Jaccard similarity** — the same method you used on text in notebook 04
- Compared the model's output against archival metadata written by human archivists — a **human spot-check** across data types

The key insight: the pipeline you learned in this workshop — extract, compare, validate — works on any unstructured data, not just text. The tools are the same. The questions you ask about trust, precision, and framing are the same.

**Next up:** The instructor will wrap up with a summary of what you did today and where to go next.